# Causal offline-to-online (Task 1): the confounded DTR

On a confounded dynamic treatment regime, the do-optimal policy (matched treatment) scores
**0.75**. A *naive* offline learner trusts the confounded logs and picks treatment 1 for both
patient subtypes — wrong for subtype 0 — asymptoting near **0.675**. The causal agents
(UC-DTR, DOVI) refuse to trust the confounded point estimates, learn online, and reach the
optimum. This is the core causal-RL lesson: confounded data isn't just noisy, it's
*misleading* — use it through causal bounds, not naively.

## The Manski causal bounds on the offline data

From the confounded logs, the interventional value of each treatment is only *partially*
identified — a `[lower, upper]` interval per (subtype, treatment).

In [1]:
from causalrl.data.dataset import generate_logs
from causalrl.envs.suite.dtr import DTREnv
from causalrl.identification.bounds import causal_q_bounds

logs = generate_logs(DTREnv(seed=100), n_episodes=4000, seed=100)
for z in (0, 1):
    for a in (0, 1):
        lo, hi = causal_q_bounds(logs, state=z, action=a)
        print(f"Z={z} a={a}: bound [{lo:.2f},{hi:.2f}] mean={logs.mean_reward(z, a):.2f}")

Z=0 a=0: bound [0.35,0.85] mean=0.70
Z=0 a=1: bound [0.41,0.90] mean=0.80
Z=1 a=0: bound [0.11,0.60] mean=0.21
Z=1 a=1: bound [0.44,0.94] mean=0.88


## Three-way comparison

UC-DTR and DOVI reach the optimum; naive-offline is stuck at the biased policy; online-only
also reaches the optimum (the offline data isn't *required* — the failure mode is *misusing*
it, which naive-offline does).

In [2]:
from causalrl.agents.baselines import NaiveOffline, OnlineOnlyUCB
from causalrl.agents.dovi import DOVI
from causalrl.agents.offline_online import UCDTR
from causalrl.eval.harness import run_episodes
from causalrl.eval.metrics import finite_horizon_regret


def evaluate(make_agent, with_offline):
    agent = make_agent()
    if with_offline:
        agent.ingest_offline(logs)
    returns = run_episodes(agent, DTREnv(seed=0), n_episodes=4000, seed=0)
    avg = sum(returns) / len(returns)
    return avg, finite_horizon_regret(returns, optimal_return=0.75)


configs = [
    ("UC-DTR", lambda: UCDTR(3, 2, seed=0), True),
    ("DOVI", lambda: DOVI(3, 2, horizon=1, seed=0), True),
    ("naive-offline", lambda: NaiveOffline(3, 2), True),
    ("online-only", lambda: OnlineOnlyUCB(3, 2, seed=0), False),
]
print(f"{'agent':16s} {'avg return':>11s} {'regret':>9s}")
for name, make, off in configs:
    avg, reg = evaluate(make, off)
    print(f"{name:16s} {avg:11.3f} {reg:9.1f}")

agent             avg return    regret
UC-DTR                 0.732      72.0
DOVI                   0.733      67.0
naive-offline          0.685     261.0
online-only            0.732      72.0


## Safety: refusing unidentifiable data

If an action was never tried in the logs, its causal bound is the vacuous `[0, 1]` — the
effect is *not identifiable* from offline data alone. In strict mode, UC-DTR refuses to build
a policy from it rather than silently trusting a guess (spec §4 "fail loudly").

In [3]:
from causalrl.data.dataset import ConfoundedTrajectoryDataset, Transition
from causalrl.exceptions import NotIdentifiableError

vacuous = ConfoundedTrajectoryDataset([Transition(0, 0, 1.0, 1, True)], n_states=2, n_actions=2)
try:
    UCDTR(2, 2, seed=0, require_identified=True).ingest_offline(vacuous)
except NotIdentifiableError as e:
    print(f"refused: {e}\n  witness (state, action) = {e.witness}")

refused: E[R|do(a=1), s=0] is not identifiable: action never logged in this state (vacuous bound [0, 1])
  witness (state, action) = (0, 1)


## v0.3: horizon-indexed DOVI on a genuinely-sequential confounded DTR

On the multi-stage `SequentialDTREnv` the do-optimal return is **1.05**. The new
finite-horizon **value-iteration** DOVI propagates value across stages and reaches it, while
the *myopic* immediate-ceiling agent (the v0.2 `DOVI(horizon=1)`) and the confounded
*naive-offline* baseline both stall near **0.85** — myopic because it can't see the
downstream consequence of its first action (the foresight gap), naive because it trusts the
confounded logs. This separates the two lessons: causal deconfounding **and** sequential
credit assignment.

In [4]:
from causalrl.data.dataset import generate_logs
from causalrl.envs.suite.seq_dtr import SequentialDTREnv

H = 2
n_states = SequentialDTREnv(horizon=H).n_states
logs = generate_logs(SequentialDTREnv(horizon=H, seed=11), n_episodes=8000, seed=11)

full = DOVI(n_states=n_states, n_actions=2, horizon=H, seed=0)
full.ingest_offline(logs)
full_late = sum(run_episodes(full, SequentialDTREnv(horizon=H, seed=0), 8000, 0)[-2000:]) / 2000

myopic = DOVI(n_states=n_states, n_actions=2, horizon=1, seed=0)  # immediate-ceiling (v0.2)
myopic.ingest_offline(logs)
myopic_late = sum(run_episodes(myopic, SequentialDTREnv(horizon=H, seed=1), 8000, 1)[-2000:]) / 2000

naive = NaiveOffline(n_states=n_states, n_actions=2)
naive.ingest_offline(logs)
naive_avg = sum(run_episodes(naive, SequentialDTREnv(horizon=H, seed=2), 8000, 2)) / 8000

print(
    f"optimal={SequentialDTREnv(horizon=H).optimal_value:.3f}  "
    f"horizon-DOVI={full_late:.3f}  myopic-DOVI={myopic_late:.3f}  naive-offline={naive_avg:.3f}"
)

optimal=1.050  horizon-DOVI=1.033  myopic-DOVI=0.869  naive-offline=0.863
